# 01 Mask Lab：Padding Mask 与 Causal Mask

目标：把 Transformer 中两种不同目的的 Mask 彻底分开。

In [1]:
import torch

PAD_ID = 0
tokens = torch.tensor([
    [5, 8, 9, 0, 0],
    [2, 3, 4, 7, 6],
])

padding_valid = tokens.ne(PAD_ID)
print(tokens)
print(padding_valid)
print("shape:", padding_valid.shape)

tensor([[5, 8, 9, 0, 0],
        [2, 3, 4, 7, 6]])
tensor([[ True,  True,  True, False, False],
        [ True,  True,  True,  True,  True]])
shape: torch.Size([2, 5])


## Padding Mask

它回答的是：**这个 Key 位置是真实 token 还是 `<pad>`？**

In [2]:
# Attention scores 假设为 [B,H,Q,K]
B, H, Q, K = 2, 3, 5, 5
scores = torch.randn(B, H, Q, K)

# [B,K] -> [B,1,1,K]
masked_scores = scores.masked_fill(
    ~padding_valid[:, None, None, :],
    float("-inf"),
)

weights = torch.softmax(masked_scores, dim=-1)
print("padding key weights sample:", weights[0, 0, 0])

padding key weights sample: tensor([0.3879, 0.4191, 0.1929, 0.0000, 0.0000])


## Causal Mask

它回答的是：**当前 Query 能不能看未来 Key？**

In [3]:
T = 6
causal = torch.ones(T, T, dtype=torch.bool).tril()
print(causal.int())

tensor([[1, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1]], dtype=torch.int32)


## 两种 Mask 可以同时存在

Decoder Self-Attention 既不能看 `<pad>`，也不能看未来。

In [4]:
valid = torch.tensor([[True, True, True, False]])  # [B,K]
causal = torch.ones(4, 4, dtype=torch.bool).tril()  # [Q,K]

combined = valid[:, None, :] & causal[None, :, :]  # [B,Q,K]
print(combined[0].int())

tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 0]], dtype=torch.int32)


### 一句话

- **Padding Mask：屏蔽“无效位置”。**
- **Causal Mask：屏蔽“未来位置”。**

两者解决的问题完全不同。